# AI Agent 개발을 위한 핵심 개념 이해

- Chain과 Agent의 결정적 차이가 **실행 경로(Path)를 누가 결정하는가**임을 설명할 수 있다.
- Agent의 **4대 핵심 구성요소**와 ReAct 루프의 **동작 및 종료 조건**을 이해한다.
- "모델은 도구를 직접 실행하지 않는다"는 사실의 소프트웨어 아키텍처적 의미를 파악한다.

---

## 0. 실행 준비

이 노트북은 `.env` 파일에 설정된 `OPENAI_API_KEY`를 사용합니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY 가 없습니다. 프로젝트 루트에 .env 를 만들고 키를 넣으세요."
)
print("준비 완료")

---

## 1. Chain 과 Agent 의 결정적 차이

이전 블록에서 다룬 RAG 파이프라인과 프롬프트 체인은 모두 **개발자가 사전에 정의한 순서대로 실행되는 Chain(결정론적 파이프라인)** 이었습니다.

- **Chain (결정론적 흐름)**: `입력 → 검색 → 프롬프트 주입 → LLM 생성`과 같이 실행 순서와 단계가 고정되어 있습니다.
- **Agent (자율적 제어 루프)**: 모델이 입력을 분석한 후, **어떤 도구를 어떤 순서로 호출할지 스스로 판단**하며 반복 실행합니다.

| 비교 항목 | Chain (단방향 파이프라인) | Agent (자율 제어 루프) |
|---|---|---|
| **경로 결정 주체** | **개발자 (코드에 고정 정의)** | **LLM (실행 중 동적 판단)** |
| **호출 횟수 및 비용** | 1회 호출 (예측 가능, 결정론적) | N회 호출 (가변적, 비결정론적) |
| **에러 전파 특성** | 파이프라인 특정 단계에서 즉시 중단 | 중간 실패 시 다른 도구로 재시도 가능 |
| **적합한 적용 영역** | 정형화된 검색/요약/번역 업무 | 다단계 추론, 도구 선택, 미지의 문제 해결 |

---

## 2. Agent 의 4대 핵심 구성요소

에이전트 시스템은 다음 4가지 핵심 요소의 상호작용으로 동작합니다.

1. **두뇌 / 모델 (Model / Brain)**: 사용자 입력과 이전 관찰 결과를 바탕으로 다음 행동을 계획하는 언어 모델
2. **도구 (Tools)**: 데이터베이스 조회, 웹 검색, 사내 API 호출 등 외부 시스템과 상호작용하는 실행 함수
3. **기억 및 상태 (Memory / State)**: 대화 이력, 중간 단계 도구 실행 결과, 상태 변수를 누적 관리하는 저장소
4. **제어 루프 (Planning & Control Loop)**: Thought(추론) → Action(도구 호출) → Observation(결과 관찰)을 반복하는 실행 엔진

---

## 3. ReAct 패러다임과 제어 루프 종료 조건

**ReAct(Reasoning + Acting)** 는 언어 모델이 추론과 행동을 번갈아 수행하며 목표를 달성하는 대표적인 에이전트 아키텍처입니다.

```mermaid
flowchart TD
    Start([사용자 입력 / State]) --> LLM["<b>Thought (추론)</b><br/>LLM 계획 수립 및 판단"]
    LLM --> Decision{"도구 호출 필요 여부<br/>(tool_calls 존재?)"}
    
    Decision -- "Yes (도구 호출)" --> Action["<b>Action (행동)</b><br/>도구 명세 생성 (tool_calls)"]
    Action --> ToolExec["<b>애플리케이션 런타임</b><br/>실제 도구 실행 (API / DB)"]
    ToolExec --> Obs["<b>Observation (관찰)</b><br/>도구 반환 결과 컨텍스트 누적"]
    Obs --> LLM
    
    Decision -- "No (종료 조건 충족)" --> FinalAns["<b>최종 응답 생성</b><br/>순수 텍스트 답변 반환"]
    FinalAns --> End([루프 종료 / Return])

    style Start fill:#f8f9fa,stroke:#6c757d
    style LLM fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    style Decision fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style Action fill:#f3e5f5,stroke:#7b1fa2
    style ToolExec fill:#e8f5e9,stroke:#388e3c
    style Obs fill:#ede7f6,stroke:#512da8
    style FinalAns fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style End fill:#eceff1,stroke:#455a64
```

- **Thought (추론)**: 현재 상태에서 필요한 다음 단계가 무엇인지 분석하고 계획을 수립합니다.
- **Action (행동)**: 필요한 도구와 전달할 인자를 결정(`tool_calls`)합니다.
- **Observation (관찰)**: 실제 시스템이 도구를 실행하고 반환한 결과를 상태에 추가하여 다음 턴의 입력으로 전달합니다.

> 🛑 **종료 조건**: 모델이 추가적인 도구 호출(`tool_calls`) 없이 **순수 텍스트(최종 답변)** 를 반환하면 ReAct 루프가 종료됩니다.

| 흔한 오해 | 실제 아키텍처 원리 |
|---|---|
| 모델이 도구를 직접 실행한다 | **모델은 도구 호출 요청(`tool_calls`) 명세만 생성합니다.** 실제 도구 실행은 애플리케이션 런타임 코드가 수행하며, 따라서 보안 및 실행 권한 통제도 애플리케이션 레벨에서 관리합니다. |

---

## 4. 실습 D5 — 다단계 추론과 모델 호출 횟수 관찰

단일 함수 호출처럼 보이는 Agent의 내부에서 **LLM API가 실제로 몇 번 호출되는지** 트레이스를 직접 확인합니다.

### 📌 실습 상황 (실제 사내 SQLite DB 연동)

사내 인사/조직 데이터베이스(`employees.db`)에 접근할 수 있는 두 개의 전용 조회 도구가 등록되어 있습니다.

| 도구 이름 | 입력 매개변수 | 조회 대상 DB 및 반환 데이터 |
|---|---|---|
| `find_employee` | 임직원 이름 (예: `"김철수"`) | `employees` 테이블 → 사번, **소속 부서**, 직급, 이메일 |
| `get_department` | 부서명 (예: `"개발팀"`) | `departments` 테이블 → 부서 ID, **팀장명**, **배정 예산** |

사용자 질문: **"김철수 님이 속한 부서의 팀장님 성함과 배정된 부서 예산을 알려줘."**

임직원 이름만으로 부서 예산과 팀장을 한 번에 조회하는 단일 테이블은 없습니다. 에이전트는 `find_employee(김철수)`로 소속 부서(`개발팀`)를 알아낸 뒤, `get_department(개발팀)`로 팀장과 예산을 조회하는 **2단계 순차 DB 조회**를 수행해야 합니다.

### 🔮 예측 — D5

**이 질문 하나를 처리하는 데 모델 API(`gpt-4.1-mini`)는 총 몇 번 호출될까요?**

① 1회 &nbsp;&nbsp;&nbsp;&nbsp; ② 2회 &nbsp;&nbsp;&nbsp;&nbsp; ③ 3회 이상

<details>
<summary>생각해보기</summary>

> **질문의 핵심:** "Agent 1회 실행 = LLM API 1회 호출"이라는 가정은 에이전트 시스템 개발 시 가장 빈번히 발생하는 오개념입니다. 비용 산정, 응답 지연(Latency), 타임아웃 설정은 모두 실제 내부 API 호출 횟수에 기반해야 합니다.

</details>

In [ ]:
import sqlite3
from pathlib import Path
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# 1. SQLite DB 경로 탐색 (미생성 시 employees.sql 시드에서 자동 복원)
DB_PATH = next(
    p / "data" / "advanced" / "employees.db"
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "advanced").is_dir()
)
if not DB_PATH.exists():
    SEED_PATH = DB_PATH.with_suffix(".sql")
    with sqlite3.connect(DB_PATH) as con:
        con.executescript(SEED_PATH.read_text(encoding="utf-8"))


# 2. 사내 인사/조직 DB 전용 도구 정의 (@tool)
@tool
def find_employee(name: str) -> str:
    """임직원 이름으로 사번, 소속 부서, 직급, 이메일을 조회합니다."""
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        cur.execute(
            "SELECT id, name, department, position, email FROM employees WHERE name = ?",
            (name,),
        )
        row = cur.fetchone()
        if not row:
            return f"'{name}' 임직원 정보를 찾을 수 없습니다."
        return f"사번: {row[0]}, 이름: {row[1]}, 부서: {row[2]}, 직급: {row[3]}, 이메일: {row[4]}"


@tool
def get_department(department_name: str) -> str:
    """부서명으로 부서 ID, 팀장명, 배정 예산을 조회합니다."""
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        cur.execute(
            "SELECT id, name, manager, budget FROM departments WHERE name = ?",
            (department_name,),
        )
        row = cur.fetchone()
        if not row:
            return f"'{department_name}' 부서 정보를 찾을 수 없습니다."
        return f"부서ID: {row[0]}, 부서명: {row[1]}, 팀장: {row[2]}, 예산: {row[3]:,}원"


# 3. 에이전트 생성 및 다단계 질의 실행
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
agent = create_agent(
    llm,
    [find_employee, get_department],
    system_prompt="사내 인사/조직 시스템 에이전트입니다. 도구로 확인한 정확한 사실만 바탕으로 답하세요.",
)

QUESTION = "김철수 님이 속한 부서의 팀장님 성함과 배정된 부서 예산을 알려줘."
out = agent.invoke({"messages": [("user", QUESTION)]})

print("[최종 응답]")
print(out["messages"][-1].content)

최종 응답이 올바르게 생성되었습니다. 이제 `out["messages"]`에 기록된 **실행 경로와 메시지 트레이스**를 확인합니다.

In [ ]:
from langchain_core.messages import AIMessage

rows = []
for i, m in enumerate(out["messages"], 1):
    kind = type(m).__name__
    if isinstance(m, AIMessage) and m.tool_calls:
        act = " · ".join(f"{c['name']}({c['args']})" for c in m.tool_calls)
    elif isinstance(m, AIMessage):
        act = "(최종 답변)"
    else:
        act = (m.content or "").strip()[:40]
    rows.append((i, kind, act))

w = max(len(k) for _, k, _ in rows)
print(f"{'#':>2}  {'메시지 타입':<{w}}  내용 / 도구 호출")
print("-" * 76)
for i, kind, act in rows:
    print(f"{i:>2}  {kind:<{w}}  {act}")

ai_turns = sum(1 for m in out["messages"] if isinstance(m, AIMessage))
print("-" * 76)
print(f"AI 턴(= 모델 API 호출) 수: {ai_turns}회   ·   전체 메시지 {len(out['messages'])}개")

> ### ▶ 함께 실행 — D5
>
> **실제 SQLite DB 조회가 필요한 다단계 질문을 실행하고 트레이스를 관찰합니다.**
>
> 🔍 **확인 사항** — `AIMessage` 개수를 확인하세요. **총 3회의 AI 턴(전체 6개 메시지)** 이 순차적으로 발생합니다.
>- 1턴: `find_employee({'name': '김철수'})` 호출 판단
>- 2턴: 조회된 부서(`개발팀`)를 바탕으로 `get_department({'department_name': '개발팀'})` 호출 판단
>- 3턴: 두 도구의 반환 결과를 종합하여 최종 응답 생성
>
> ⚠️ **참고**: LLM 생성 특성에 따라 도구 인자 포맷이나 호출 횟수에 미세한 차이가 발생할 수 있습니다.

---

### 📊 결과 해석

**예상되는 결과:** `AIMessage`가 총 3회 발생합니다 (① `find_employee` 호출 판단 → ② `get_department` 호출 판단 → ③ 최종 답변 생성). 즉, 단 한 번의 `agent.invoke()` 실행에 **3회의 LLM API 호출**이 발생했습니다.

| 결과 상황 | 해설 및 원인 분석 |
|---|---|
| 예상대로 출력됨 (AI 턴 3회) | 애플리케이션 코드는 `agent.invoke()` 1회만 호출했으나, 모델 내부적으로 **DB 도구 탐색 2회 + 최종 응답 1회**가 발생했습니다. 에이전트 사용 시 호출 횟수는 코드가 아닌 모델의 판단에 의해 결정됩니다. |
| AI 턴이 4회 이상 발생한 경우 | 모델이 도구 반환값을 확인한 후 추가 검증을 위해 도구를 재호출한 케이스입니다. 비결정론적 특성으로 인해 동일한 질문이라도 실행 시점마다 호출 횟수가 달라질 수 있습니다. |
| AI 턴이 1회로 종료된 경우 | 도구를 호출하지 않고 모델의 사전 지식으로 임의 답변을 생성한 경우입니다 (환각/도구 미호출). 시스템 프롬프트에 도구 사용 필수 조건을 강화해야 합니다. |

> 🎯 **핵심 아키텍처 원칙**: **Agent 1회 실행 ≠ LLM API 1회 호출**
>
> 애플리케이션에서는 단 한 번의 `invoke()`를 실행하더라도, 내부에서는 목표 달성을 위해 모델이 여러 차례 연속적으로 호출됩니다. 따라서 에이전트 시스템을 설계할 때는 **최대 루프 상한(Iteration Cap)** 과 **비용/지연 예산**을 반드시 코드로 제어해야 합니다.

| 흔한 오해 | 실제 아키텍처 원리 |
|---|---|
| Agent 실행 비용은 Chain과 유사하다 | 에이전트는 다단계 루프를 돌며 컨텍스트가 누적되므로, 체인에 비해 **호출 횟수와 토큰 소비량이 수 배 이상 증가**할 수 있습니다. |

---

## 5. 실무 관점 — Agent 도입 시 발생하는 주요 위험 요소

In [ ]:
import sqlite3
from pathlib import Path
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# 1. SQLite DB 경로 탐색 (미생성 시 inventory.sql 시드에서 자동 복원)
DB_PATH = next(
    p / "data" / "advanced" / "inventory.db"
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "advanced").is_dir()
)
if not DB_PATH.exists():
    SEED_PATH = DB_PATH.with_suffix(".sql")
    with sqlite3.connect(DB_PATH) as con:
        con.executescript(SEED_PATH.read_text(encoding="utf-8"))

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# D7 실습: 도구 설명(Docstring)이 모호할 때 vs 명확할 때의 DB 연동 도구 선택 비교
QUESTION_D7 = "노트북 재고 몇 대 남았어?"


# 1. 설명 부실 (Docstring이 모호하여 키워드 매칭 오류 유발 -> 잘못된 DB 테이블 조회)
@tool
def query_supply_db_vague(query: str) -> str:
    """재고 DB를 조회합니다."""
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        cur.execute(
            "SELECT item_name, stock_count FROM office_supplies WHERE item_name LIKE ?",
            (f"%{query}%",),
        )
        rows = cur.fetchall()
        if not rows:
            return f"[소모품 DB] \x27{query}\x27 검색 결과가 없습니다."
        return "[소모품 DB 결과] " + ", ".join(
            [f"{name}: {cnt}개" for name, cnt in rows]
        )


@tool
def query_asset_db_vague(query: str) -> str:
    """자산 DB를 조회합니다."""
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        cur.execute(
            "SELECT asset_name, available_count FROM it_assets WHERE asset_name LIKE ?",
            (f"%{query}%",),
        )
        rows = cur.fetchall()
        if not rows:
            return f"[IT자산 DB] \x27{query}\x27 검색 결과가 없습니다."
        return "[IT자산 DB 결과] " + ", ".join(
            [f"{name}: {cnt}대" for name, cnt in rows]
        )


tools_vague = [query_supply_db_vague, query_asset_db_vague]
resp_vague = llm.bind_tools(tools_vague).invoke(QUESTION_D7)
tool_call_vague = resp_vague.tool_calls[0] if resp_vague.tool_calls else None


# 2. 설명 명확 (Docstring에 구체적 대상과 제약 조건 명시 -> 정확한 DB 테이블 조회)
@tool
def query_office_supplies_db(item_keyword: str) -> str:
    """사무용품, 문구류, 소모품(복사용지, 볼펜, 포스트잇 등)의 재고 수량을 조회합니다. IT 전산 자산에는 사용하지 마세요."""
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        cur.execute(
            "SELECT item_name, stock_count FROM office_supplies WHERE item_name LIKE ?",
            (f"%{item_keyword}%",),
        )
        rows = cur.fetchall()
        if not rows:
            return f"[소모품 DB] \x27{item_keyword}\x27 검색 결과가 없습니다."
        return "[소모품 DB 결과] " + ", ".join(
            [f"{name}: {cnt}개" for name, cnt in rows]
        )


@tool
def query_it_assets_db(device_keyword: str) -> str:
    """노트북, 모니터 등 IT 전산 자산의 보유 수량을 조회합니다."""
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        search_term = "MacBook" if "노트북" in device_keyword else device_keyword
        cur.execute(
            "SELECT asset_name, available_count FROM it_assets WHERE asset_name LIKE ?",
            (f"%{search_term}%",),
        )
        rows = cur.fetchall()
        if not rows:
            return f"[IT자산 DB] \x27{device_keyword}\x27 검색 결과가 없습니다."
        return "[IT자산 DB 결과] " + ", ".join(
            [f"{name}: {cnt}대" for name, cnt in rows]
        )


tools_clear = [query_office_supplies_db, query_it_assets_db]
resp_clear = llm.bind_tools(tools_clear).invoke(QUESTION_D7)
tool_call_clear = resp_clear.tool_calls[0] if resp_clear.tool_calls else None

print(f"질문: \"{QUESTION_D7}\"")
if tool_call_vague:
    name_v = tool_call_vague["name"]
    args_v = tool_call_vague["args"]
    func_vague = {t.name: t for t in tools_vague}[name_v]
    print(f"  [설명 부실] 선택된 도구: {name_v} -> DB 실행 결과: {func_vague.invoke(args_v)}")
else:
    print("  [설명 부실] 도구 미호출")

if tool_call_clear:
    name_c = tool_call_clear["name"]
    args_c = tool_call_clear["args"]
    func_clear = {t.name: t for t in tools_clear}[name_c]
    print(f"  [설명 명확] 선택된 도구: {name_c}   -> DB 실행 결과: {func_clear.invoke(args_c)}")
else:
    print("  [설명 명확] 도구 미호출")


In [ ]:
import sqlite3
from pathlib import Path
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

# D7-보조 실습: 도구 이름(Name) 및 질의 키워드에 따른 도구 선택/오호출 궤적 관찰
# (실제 Tavily Web Search 도구와 SQLite 사내 DB 연동)

# 1. 실제 Tavily 검색 도구 준비
tavily_search = TavilySearch(max_results=2)
web_search_tool = StructuredTool.from_function(
    func=lambda query: tavily_search.invoke({"query": query}),
    name="web_search",
    description="인터넷에서 최신 공개 정보를 검색한다.",
)

# 2. 실제 사내 SQLite DB 연동 함수 (미생성 시 시드에서 자동 복원)
EMP_DB_PATH = next(
    p / "data" / "advanced" / "employees.db"
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "advanced").is_dir()
)
if not EMP_DB_PATH.exists():
    with sqlite3.connect(EMP_DB_PATH) as con:
        con.executescript(EMP_DB_PATH.with_suffix(".sql").read_text(encoding="utf-8"))

def query_dept_budget_db(query: str) -> str:
    with sqlite3.connect(EMP_DB_PATH) as con:
        cur = con.cursor()
        cur.execute("SELECT name, budget FROM departments")
        rows = cur.fetchall()
        return "[사내 DB 부서 예산] " + ", ".join([f"{name}: {budget:,}원" for name, budget in rows])


def create_tools_with_db_name(db_name: str):
    """동일한 Docstring('데이터를 조회한다.')을 가지되 도구 이름(Name)만 다르게 등록"""
    db_tool = StructuredTool.from_function(
        func=query_dept_budget_db,
        name=db_name,
        description="데이터를 조회한다.",  # 두 조건 모두 완전히 동일한 설명
    )
    return [db_tool, web_search_tool]


# 실습 테스트 케이스: 정상 선택 vs 이름 뉘앙스 vs 외부 검색 키워드 유인으로 인한 오호출 실패
test_scenarios = [
    (
        "우리 회사 부서별 예산 현황 알려줘",
        "internal_db",
        "[케이스 1] 도구명 직관적 (internal_db)",
        "이름의 뉘앙스를 즉각 인식하여 사내 DB 선택",
    ),
    (
        "우리 회사 부서별 예산 현황 알려줘",
        "source_b",
        "[케이스 2] 도구명 모호함 (source_b)",
        "이름 단서가 없어 소거법 비교를 거쳐 선택",
    ),
    (
        "구글에서 우리 회사 부서별 예산 현황 검색해서 찾아줘",
        "source_b",
        "[케이스 3] 외부 검색 키워드 유인 실패 케이스",
        "\x27구글/검색\x27 키워드에 유인되어 web_search 오호출 -> 사내 정보 조회 실패",
    ),
]

for question, db_name, title, desc in test_scenarios:
    tools = create_tools_with_db_name(db_name)
    tool_names = [t.name for t in tools]
    resp = llm.bind_tools(tools).invoke(question)
    
    tool_calls = resp.tool_calls
    call_count = len(tool_calls)
    pick = tool_calls[0] if tool_calls else None
    name = pick["name"] if pick else "도구 미호출"
    args = pick["args"] if pick else {}
    usage = resp.usage_metadata or {}
    in_tok = usage.get("input_tokens", "-")
    out_tok = usage.get("output_tokens", "-")
    
    print(f"=== {title} ===")
    print(f"  • 사용자 질문      : \"{question}\"")
    print(f"  • 등록 도구 목록   : {tool_names} (설명: \x27데이터를 조회한다.\x27)")
    print(f"  • 호출 횟수 / 토큰 : {call_count}회 호출 (입력 {in_tok} 토큰 / 출력 {out_tok} 토큰)")
    print(f"  • 선택된 도구(Call): {name}(args={args})")
    
    if pick:
        func = {t.name: t for t in tools}[name]
        result = func.invoke(args)
        if name == "web_search":
            print(f"  • 실제 DB 실행결과 : [웹 검색 실행됨 - 외부망에서는 사내 예산 조회 불가] (❌ 실패)")
        else:
            print(f"  • 실제 DB 실행결과 : {result} (✅ 성공)")
    else:
        print(f"  • 실제 DB 실행결과 : 도구가 호출되지 않음 (❌ 실패)")
    print(f"  • 판정 해설        : {desc}\n")


### 📊 실습 결과 상세 분석 — Agent 동작을 결정하는 3가지 실무 원칙

위 실습 코드(D5, D7, D7-보조)에서 관찰된 동작은 실무 에이전트 개발 시 반드시 지켜야 할 아키텍처 원리를 보여줍니다.

---

#### 1. `[D5]` 다단계 질문의 모델 호출 횟수 (AI 턴 수 N회)
- **관찰 결과**: 2단계 도구 호출(`find_employee` → `get_department`) 질문에서 **총 3회의 LLM API 호출** 발생
- **아키텍처 의미**: 단 한 번의 `agent.invoke()` 실행이라도 내부적으로 **① 1차 도구 호출 판단 → ② 2차 도구 호출 판단 → ③ 최종 텍스트 응답 생성**의 3단계를 순차적으로 거칩니다.
- **실무 교훈**: 에이전트의 응답 지연 시간(Latency)과 비용 예산은 `사용자 질문 수 × 1`이 아니라 `예상 평균 도구 호출 스텝 수(N)`를 곱하여 산정해야 합니다.

---

#### 2. `[D7]` 도구 설명(Docstring)에 따른 DB 조회 정밀도
- **관찰 결과**:
  - **설명 부실(`"재고 DB를 조회합니다."` 등 모호)**: 모델이 단순 키워드("재고")에 이끌려 소모품 테이블 조회 도구(`query_supply_db_vague`)를 선택하여 엉뚱한 테이블을 조회 (`[소모품 DB] 검색 결과 없음` 반환)
  - **설명 명확(도구의 대상 및 배제 조건 명시)**: 모델이 문맥(노트북 = 전산 기기)을 파악하고 정확한 IT 자산 DB 도구(`query_it_assets_db`)를 호출하여 실제 재고(`MacBook Pro 16: 12대`) 조회 성공
- **아키텍처 의미**: `@tool` 함수의 **Docstring은 단순 주석이 아니라, 모델이 어떤 DB 테이블/API를 호출할지 결정하는 핵심 시스템 프롬프트(Tool Schema)** 입니다.
- **실무 교훈**: 도구 호출 오류나 잘못된 DB 조회가 발생할 때 라우팅 코드를 수정하기 전에 **도구 설명문(Docstring)의 역할 정의, 대상 테이블, 배제 조건(예: "IT 기기에는 사용 금지")을 명확히 명시**해야 합니다.

---

#### 3. `[D7-보조]` 도구 식별자 명칭(Name)의 의미적 편향과 키워드 유인 실패 (이름도 프롬프트다)
- **관찰 결과**:
  - **케이스 1 (`internal_db`)**: 도구 설명이 모호하더라도 이름의 뉘앙스(`internal`)를 즉각 인식하여 사내 DB 선택 및 실제 예산 조회 성공 (✅)
  - **케이스 2 (`source_b`)**: 이름에 의미 단서가 없어 소거법 비교를 거치며 판단이 진행됨 (✅)
  - **케이스 3 (키워드 유인 실패)**: 사용자 질문에 `"구글에서 검색해줘"` 등 외부 키워드가 섞이자, 사내 데이터 질의임에도 모델이 키워드에 유인되어 **`web_search`를 오호출하여 실패 (❌)**
- **아키텍처 의미**: LLM에게는 도구 식별자 이름(`name`)과 사용자 프롬프트 속 유인 키워드가 도구 라우팅 결정에 막대한 영향을 미칩니다.
- **실무 교훈**: 직관적인 도구명 명명이 필수적이며, **사내 질의 처리 시 외부 웹 검색 오호출을 원천 차단하려면 프롬프트 지시가 아닌 런타임 도구 목록(`tools`)에서 `web_search`를 완전히 제외(도구 목록 동적 필터링)** 해야 합니다.

| 흔한 오해 | 실제 아키텍처 원리 |
|---|---|
| 프롬프트 지시만으로 특정 도구 사용을 완벽히 금지할 수 있다 | 도구 목록에 노출되어 있으면 모델이 임의로 호출할 가능성이 상존합니다. **도구 접근 제어는 프롬프트가 아닌 코드(도구 목록 동적 필터링)로 구현**해야 합니다. |

In [ ]:
# 기본 컨텍스트 (시스템 프롬프트 및 도구 스키마 - 매 스텝 고정 재전송)
BASE = 500
# 스텝당 누적되는 대화 및 추론 토큰
TURN = 120


def simulate(steps: int, observation: int) -> list[int]:
    """스텝별 누적 컨텍스트(입력 토큰)를 계산합니다."""
    ctx, per_step = BASE, []
    for _ in range(steps):
        per_step.append(ctx)
        ctx += TURN + observation  # 이전 스텝의 관찰 결과가 다음 스텝 입력으로 누적됨
    return per_step


for label, steps, obs in [
    ("3스텝 · 관찰 결과 작음 (200토큰)", 3, 200),
    ("6스텝 · 관찰 결과 작음 (200토큰)", 6, 200),
    ("3스텝 · 관찰 결과 큼 (2000토큰)", 3, 2000),
    ("6스텝 · 관찰 결과 큼 (2000토큰)", 6, 2000),
]:
    per = simulate(steps, obs)
    print(f"{label}\n    스텝별 입력 토큰: {per}\n    총 누적 입력 토큰: {sum(per):,} 토큰\n")

a = sum(simulate(3, 200))
b = sum(simulate(6, 200))
c = sum(simulate(3, 2000))
print("-" * 64)
print(f"스텝 수 2배 증가 (3→6스텝, 관찰 200)  : {a:,} → {b:,} 토큰 ({b/a:.1f}배 증가)")
print(f"관찰 크기 10배 증가 (3스텝, 200→2000): {a:,} → {c:,} 토큰 ({c/a:.1f}배 증가)")
print("\n→ 스텝 수를 늘리지 않아도, 도구의 반환 데이터(관찰 크기)가 커지면 비용이 급격히 증가합니다.")

스텝 수를 2배로 늘린 경우(3.2배)와 **스텝 수는 고정한 채 도구의 관찰 결과 크기만 키운 경우**(3.2배)가 거의 동일한 수준으로 토큰 소비를 증가시킵니다.

> 🎯 **도구 설계 원칙**: 도구는 전체 원본 데이터를 그대로 반환하지 말고, **모델 판단에 꼭 필요한 필드만 정제/요약하여 반환**하도록 구현해야 합니다. 방대한 데이터를 반환하는 도구는 해당 호출 시점뿐만 아니라 **후속하는 모든 스텝의 입력 토큰에 누적 과금**됩니다.

### 🛡️ 실무 해결책: LangChain `create_agent`의 미들웨어(Middleware)로 컨텍스트 압축

관찰 결과가 거대해지거나 다단계 대화가 길어질 때 발생하는 **컨텍스트 누적 과금**과 **무한 루프 폭주**는 LangChain의 내장 미들웨어(`SummarizationMiddleware`, `ModelCallLimitMiddleware`)를 통해 프로덕션 수준으로 방어할 수 있습니다.

In [ ]:
import sqlite3
from pathlib import Path
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, ModelCallLimitMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 사내 긴 문서/데이터 조회 도구 예시
@tool
def fetch_company_policy(topic: str) -> str:
    """사내 복지 및 출장 규정 상세 문서를 조회합니다."""
    return f"[{topic} 사내 규정 전문] " + (
        "제1조(목적) 본 규정은 임직원의 복지 및 여비 지급 기준을 규정함을 목적으로 한다. "
        "제2조(지급기준) 국내 출장 일비 5만원, 숙박비 실비 정산(상한 10만원). " * 20
    )


# LangChain create_agent에 내장 미들웨어 스택 적용
agent_with_middleware = create_agent(
    model=llm,
    tools=[fetch_company_policy],
    system_prompt="사내 규정 도우미 에이전트입니다. 도구로 확인한 사실만 간결히 요약해 답하세요.",
    middleware=[
        # 1) 컨텍스트 압축: 토큰/메시지 임계치 초과 시 과거 관찰 결과 자동 요약 압축
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 2000),  # 컨텍스트 2000 토큰 초과 시 자동 요약 트리거
            keep=("messages", 2),      # 최근 2개 메시지는 원본 유지
        ),
        # 2) 실행 비용/루프 상한: 최대 호출 횟수 강제 제한 (무한 루프 과금 방지)
        ModelCallLimitMiddleware(run_limit=5),
    ],
)

res = agent_with_middleware.invoke({
    "messages": [("user", "출장 여비 지급 기준 규정 전문을 조회해서 핵심만 알려줘.")]
})

messages = res["messages"]
final_msg = messages[-1]
usage = getattr(final_msg, "usage_metadata", {}) or {}
in_tok = usage.get("input_tokens", "-")
out_tok = usage.get("output_tokens", "-")
total_chars = sum(len(m.content) for m in messages if isinstance(m.content, str))

print("[미들웨어가 제어한 최종 응답]")
print(final_msg.content)
print("\n" + "=" * 64)
print(f"• 최종 응답 글자수 : {len(final_msg.content):,}자")
print(f"• 전체 메시지 글자수: {total_chars:,}자 (누적 메시지 {len(messages)}개)")
print(f"• 최종 단계 토큰    : 입력 {in_tok} 토큰 / 출력 {out_tok} 토큰")
print("=" * 64)


| 흔한 오해 | 실제 아키텍처 원리 |
|---|---|
| Agent 비용은 단순 스텝 수에 선형 비례한다 | 비용은 **누적 컨텍스트 크기**에 의해 결정됩니다. 큰 관찰 결과 하나가 후속 스텝 전체의 비용을 기하급수적으로 증가시킵니다. |

---

## 6. 단순 ReAct 를 넘어 LangGraph 가 필요한 이유

단순 ReAct 루프는 구현이 간편하지만, 엔터프라이즈 환경에서는 다음과 같은 구조적 한계에 부딪힙니다.

| 구분 | 단순 ReAct (블랙박스 루프) | LangGraph (상태 기반 워크플로우 그래프) |
|---|---|---|
| **흐름 제어** | 모델 자율 루프 (제어 및 중단 어려움) | **노드와 조건부 엣지**를 통한 명시적 경로 제어 |
| **상태 관리** | 메시지 리스트 단순 누적 | **State 스키마 및 리듀서(Reducer)** 기반 정밀 제어 |
| **사람 개입 (HITL)** | 런타임 인터럽트 지원 부재 | `interrupt()`를 통한 **승인 게이트 및 상태 복원** 지원 |
| **지속성 및 복구** | 메모리 기반 (프로세스 종료 시 유실) | **체크포인터(Checkpointer)** 기반 실행 상태 영속화 |
| **병렬 및 분기** | 단일 순차 실행 | **`Send` API를 통한 동적 병렬 맵-리듀스** 지원 |

---

## 정리

- Chain과 Agent의 차이는 기능의 우열이 아니라 **실행 경로를 누가 결정하는가**에 있습니다.
- ReAct 에이전트의 종료 조건은 `tool_calls`가 없는 최종 텍스트 생성입니다. (무한 루프 방지를 위해 상한을 코드로 설정 필수)
- 모델은 도구를 **실행하지 않고 호출 요청만 생성**합니다. 실제 실행과 권한 통제는 애플리케이션 코드의 영역입니다.
- **Agent 1회 실행 ≠ LLM 1회 호출**입니다. 다단계 도구 호출 시 여러 번의 API 호출이 발생합니다.
- 도구 접근 통제는 프롬프트 지시가 아닌 **도구 목록(`tools`) 동적 필터링**으로 구현합니다.
- 에이전트 비용은 스텝 수뿐만 아니라 **도구가 반환하는 관찰 데이터 크기**에 크게 좌우됩니다.
- 고신뢰성 에이전트 설계를 위해 업무에 필요한 **최소 자율성 수준**을 정의하고, LangGraph를 통해 안정적인 제어 흐름을 구현합니다.